In [ ]:
"""
=============================================================================
PRE-MODELING 1: persistence of zeros (history rule for the "sin deserción observada" group)
=============================================================================
Question answered: if a sede shows NO dropout in its own history (DESERCION_LAG3 == 0), how often does
it still have a positive dropout rate in the target year? That share is the "false reassurance" rate of
the rule used by the tool to label sedes as "sin deserción observada".

The rule has no parameters (nothing is fitted), so it is evaluated on every eligible row, by target year,
split and segment. Two versions are reported:
  - A: history zero = DESERCION_LAG3 == 0 (includes the imputed zeros, i.e. sedes without Sit_Acad record)
  - B: history zero = DESERCION_LAG3 == 0 and the zero was observed (not imputed)

Input : Data/Processed/panel_final_<panel>.csv
Output: Data/Auditorio_Panel_Final/pre_modelo_persistencia_<panel>.csv and .md
=============================================================================
"""

from pathlib import Path

import numpy as np
import pandas as pd

# =============================================================================
# 0. PATHS
# =============================================================================
BASE_DIR = Path(
    r"C:\Users\DELL\OneDrive\Escritorio\UNIVERSIDAD\Maestria Business A\Proyecto Empresarial\Data"
)
IN_DIR = BASE_DIR / "Processed"
AUDIT_DIR = BASE_DIR / "Auditorio_Panel_Final"
PANELES = ["ofrecen11", "Solo_Primaria"]

# =============================================================================
# 1. GENERAL VARIABLES
# =============================================================================
COL_SEDE = "SEDE_CODIGO"
COL_ANIO = "ANIO_OBJETIVO"
COL_SEGMENTO = "SEGMENTO_MODELO"
COL_TARGET = "TASA_DESERCION_SEDE"
COL_HIST = "DESERCION_LAG3"
COL_HIST_IMPUTADA = "FLAG_DESERCION_LAG3_CERO_IMPUTADO"
USE_COLS = [
    COL_SEDE, COL_ANIO, "SPLIT", COL_SEGMENTO, "ELEGIBLE_MODELO", "TIENE_HIST_DESERCION",
    COL_HIST, COL_HIST_IMPUTADA, COL_TARGET, "ALTO_RIESGO",
]
VERSIONES = {
    "A_incluye_ceros_imputados": lambda d: d[COL_HIST] == 0,
    "B_solo_ceros_observados": lambda d: (d[COL_HIST] == 0) & (d[COL_HIST_IMPUTADA] == 0),
}

# =============================================================================
# 2. FUNCTION DEFINITIONS
# =============================================================================

def summarize(d: pd.DataFrame, in_rule: pd.Series) -> dict:
    """Metrics of the history rule on the rows of d (all of them have history and are eligible)."""
    positive = d[COL_TARGET] > 0
    alto = d["ALTO_RIESGO"].fillna(0).astype(int) == 1
    n, n_rule = len(d), int(in_rule.sum())
    out_rule = ~in_rule
    return {
        "FILAS": n,
        "FILAS_EN_REGLA": n_rule,
        "PCT_EN_REGLA": 100 * n_rule / n if n else np.nan,
        # Share of the rule group that still shows dropout: the false reassurance of the tool
        "PCT_FALSA_TRANQUILIDAD": 100 * float(positive[in_rule].mean()) if n_rule else np.nan,
        # Same share outside the rule, to read how much the history separates the two groups
        "PCT_POSITIVO_FUERA_REGLA": 100 * float(positive[out_rule].mean()) if out_rule.any() else np.nan,
        # Positives and PRIORITARIO sedes that the rule would label as "sin deserción observada"
        "PCT_POSITIVOS_EN_REGLA": 100 * float((positive & in_rule).sum() / positive.sum()) if positive.sum() else np.nan,
        "PCT_PRIORITARIO_EN_REGLA": 100 * float((alto & in_rule).sum() / alto.sum()) if alto.sum() else np.nan,
    }


def evaluate_panel(df: pd.DataFrame, nombre: str) -> pd.DataFrame:
    """Evaluate both versions of the rule by target year, split and segment."""
    base = df[(df["ELEGIBLE_MODELO"] == 1) & (df["TIENE_HIST_DESERCION"] == 1)].copy()
    filas = []
    for version, regla in VERSIONES.items():
        in_rule = regla(base)
        cortes = [
            ("TOTAL", lambda d: pd.Series("todas", index=d.index)),
            ("ANIO_OBJETIVO", lambda d: d[COL_ANIO].astype(int).astype(str)),
            ("SPLIT", lambda d: d["SPLIT"]),
            ("SEGMENTO", lambda d: d[COL_SEGMENTO]),
        ]
        for tipo, clave in cortes:
            grupos = clave(base)
            for g, idx in base.groupby(grupos).groups.items():
                fila = {"PANEL": nombre, "VERSION_REGLA": version, "CORTE": tipo, "GRUPO": g}
                fila.update(summarize(base.loc[idx], in_rule.loc[idx]))
                filas.append(fila)
    return pd.DataFrame(filas)


def to_markdown(df: pd.DataFrame) -> str:
    """Minimal markdown table (no extra dependency)."""
    cols = list(df.columns)
    lines = ["| " + " | ".join(cols) + " |", "| " + " | ".join(["---"] * len(cols)) + " |"]
    for r in df.itertuples(index=False):
        lines.append("| " + " | ".join(f"{v:.2f}" if isinstance(v, float) else str(v) for v in r) + " |")
    return "\n".join(lines)


def save_report(tabla: pd.DataFrame, nombre: str) -> None:
    """Write csv and md; a locked OneDrive file must not abort the analysis."""
    try:
        AUDIT_DIR.mkdir(parents=True, exist_ok=True)
        tabla.to_csv(AUDIT_DIR / f"pre_modelo_persistencia_{nombre}.csv", index=False)
        resumen = tabla[tabla["CORTE"].isin(["TOTAL", "ANIO_OBJETIVO", "SEGMENTO"])]
        md = [f"# Persistencia de ceros - panel {nombre}", "", to_markdown(resumen)]
        (AUDIT_DIR / f"pre_modelo_persistencia_{nombre}.md").write_text("\n".join(md), encoding="utf-8")
    except OSError as err:
        print(f"[WARN] Could not write the report of {nombre}: {err}")


# =============================================================================
# 3. EXECUTION
# =============================================================================

def main() -> None:
    for nombre in PANELES:
        path = IN_DIR / f"panel_final_{nombre}.csv"
        df = pd.read_csv(path, usecols=USE_COLS, dtype={COL_SEDE: str}, low_memory=False)
        tabla = evaluate_panel(df, nombre)
        save_report(tabla, nombre)

        print(f"\n=== {nombre}: false reassurance of the history rule (eligible rows with history) ===")
        vista = tabla[(tabla["CORTE"].isin(["TOTAL", "ANIO_OBJETIVO"]))][
            ["VERSION_REGLA", "CORTE", "GRUPO", "FILAS", "PCT_EN_REGLA", "PCT_FALSA_TRANQUILIDAD",
             "PCT_POSITIVO_FUERA_REGLA", "PCT_POSITIVOS_EN_REGLA", "PCT_PRIORITARIO_EN_REGLA"]
        ]
        print(vista.round(1).to_string(index=False))


if __name__ == "__main__":
    main()



=== ofrecen11: false reassurance of the history rule (eligible rows with history) ===
            VERSION_REGLA         CORTE GRUPO  FILAS  PCT_EN_REGLA  PCT_FALSA_TRANQUILIDAD  PCT_POSITIVO_FUERA_REGLA  PCT_POSITIVOS_EN_REGLA  PCT_PRIORITARIO_EN_REGLA
A_incluye_ceros_imputados         TOTAL todas  49150          39.8                    28.8                      73.2                    20.6                      21.9
A_incluye_ceros_imputados ANIO_OBJETIVO  2020  12158          37.5                    30.5                      73.4                    20.0                      19.4
A_incluye_ceros_imputados ANIO_OBJETIVO  2021  12214          39.7                    29.3                      76.8                    20.0                      19.6
A_incluye_ceros_imputados ANIO_OBJETIVO  2022  12381          39.7                    26.6                      74.9                    19.0                      19.1
A_incluye_ceros_imputados ANIO_OBJETIVO  2023  12397          42.4            